# 11. NumPy Deep Dive -- Arrays for Science

In notebooks 05 and 06 we introduced NumPy arrays and basic indexing. In this notebook we go deeper into the operations that make NumPy the backbone of scientific computing in Python.

**Why does this matter for the Kaggle challenge?** Machine learning algorithms expect NumPy arrays as input. To prepare our polymer data for modelling, we need to reshape, combine, and transform arrays fluently.

**Topics covered:**
- Array shapes and reshaping
- Aggregation along axes (`mean`, `std`, `sum`, `argmax`, ...)
- Broadcasting
- Stacking and concatenation
- Matrix multiplication

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 11.1 From DataFrames to Arrays

In CodingTask 1 you built a feature table for the polymer dataset and saved it as `artifacts/ffv_features_basic.csv`. To make this notebook self-contained, the cell below **rebuilds that file from `data/train.csv` if it is missing**. If you already completed CodingTask 1, nothing happens -- your existing file is kept.

In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
features_path = ARTIFACT_DIR / "ffv_features_basic.csv"

if not features_path.exists():
    print("Rebuilding artifacts/ffv_features_basic.csv from data/train.csv ...")
    train_df = pd.read_csv("data/train.csv")

    # Keep only rows with an FFV value, and only the columns we need
    ffv_df = train_df.dropna(subset=["FFV"])[["id", "SMILES", "FFV"]].copy()

    # Basic SMILES features (same as CodingTask 1)
    ffv_df["smiles_len"]  = ffv_df["SMILES"].str.len()
    ffv_df["count_star"]  = ffv_df["SMILES"].str.count(r"\*")
    ffv_df["count_C"]     = ffv_df["SMILES"].str.count("C")
    ffv_df["count_O"]     = ffv_df["SMILES"].str.count("O")
    ffv_df["count_N"]     = ffv_df["SMILES"].str.count("N")
    ffv_df["count_equal"] = ffv_df["SMILES"].str.count("=")

    ffv_df.to_csv(features_path, index=False)
    print(f"Saved {features_path} with {len(ffv_df)} rows.")
else:
    print(f"Found existing {features_path} -- skipping rebuild.")

In [ ]:
df = pd.read_csv("artifacts/ffv_features_basic.csv")
df.head()

A DataFrame is essentially a labelled wrapper around a NumPy array. We can extract the raw array with `.values`:

In [ ]:
feature_cols = ["smiles_len", "count_star", "count_C", "count_O", "count_N", "count_equal"]
features = df[feature_cols].values

print("Type:", type(features))
print("Shape:", features.shape)   # (rows, columns)
print("Dtype:", features.dtype)
print("Ndim: ", features.ndim)

So our feature matrix is a 2D array with **7030 rows** (polymers) and **6 columns** (features). This is the format that machine learning expects.

---
## 11.2 Reshaping and Dimensions

NumPy arrays can have any number of dimensions:
- **1D** -- a flat sequence (e.g. a list of FFV values)
- **2D** -- a table / matrix (e.g. our feature matrix)
- **3D** -- a stack of matrices (e.g. a colour image: height x width x 3)

We can change the shape of an array **without changing its data** using `reshape`.

In [ ]:
# Start with a simple 1D array
a = np.arange(12)
print("1D:", a, "  shape:", a.shape)

In [ ]:
# Reshape into a 2D array (3 rows, 4 columns)
b = a.reshape(3, 4)
print(b)
print("Shape:", b.shape)

In [ ]:
# Reshape into 3D (2 x 2 x 3)
c = a.reshape(2, 2, 3)
print(c)
print("Shape:", c.shape)

### The `-1` shortcut

If you set one dimension to `-1`, NumPy figures it out automatically:

In [ ]:
# "I want 4 columns, you figure out the rows"
d = a.reshape(-1, 4)
print(d)
print("Shape:", d.shape)

### Going back to 1D

`flatten()` and `ravel()` both return a 1D view of the data:

In [ ]:
print(b.flatten())
print(b.ravel())

> **Chemistry hook:** A 200x200 grayscale image of a molecule is a 2D array of shape `(200, 200)`.  
> A colour image is 3D: `(200, 200, 3)` -- one layer for Red, Green, Blue.  
> We will see this in Notebook 12 when we draw polymer structures with RDKit!

### Exercise 1

1. Create an array containing the numbers 0 to 23 (hint: `np.arange(24)`).
2. Reshape it to `(4, 6)`.
3. Reshape it to `(2, 3, 4)`.
4. Flatten it back to 1D and verify you get the original array.

In [ ]:
# Your code here


In [ ]:
a = np.arange(24)
print("1D:", a, "shape:", a.shape)

b = a.reshape(4, 6)
print("\n(4, 6):\n", b)

c = a.reshape(2, 3, 4)
print("\n(2, 3, 4):\n", c)

flat = c.flatten()
print("\nFlattened equals original?", np.array_equal(flat, a))

---
## 11.3 Aggregation Along Axes

NumPy provides functions like `np.mean`, `np.sum`, `np.std`, `np.min`, `np.max`, `np.argmin`, `np.argmax`. When applied to a 2D array, the `axis` parameter controls **which dimension gets collapsed**:

- `axis=0` -- collapse rows -> one result **per column** (per feature)
- `axis=1` -- collapse columns -> one result **per row** (per polymer)

Think of it as: the axis you specify **disappears** from the result.

In [ ]:
# Mean of each feature across all polymers
feature_means = np.mean(features, axis=0)
print("Mean per feature:", feature_means)
print("Shape:", feature_means.shape)  # (6,) -- one value per column

In [ ]:
# Mean across features for each polymer
polymer_means = np.mean(features, axis=1)
print("Mean per polymer (first 10):", polymer_means[:10])
print("Shape:", polymer_means.shape)  # (7030,) -- one value per row

In [ ]:
# Standard deviation per feature
feature_stds = np.std(features, axis=0)
print("Std per feature:", feature_stds)

In [ ]:
# Which feature has the highest variance?
variances = np.var(features, axis=0)
print("Variances:", variances)
print("Feature with highest variance:", feature_cols[np.argmax(variances)])

### Exercise 2

1. Compute the **minimum** and **maximum** of each feature column.
2. Which polymer (row index) has the **longest SMILES string**? (Hint: `smiles_len` is the first column, use `np.argmax`).
3. Compute the **sum** of each feature for the first 100 polymers only (use slicing + `np.sum`).

In [ ]:
# Your code here


In [ ]:
# 1. min and max of each feature column
print("Min per feature:", np.min(features, axis=0))
print("Max per feature:", np.max(features, axis=0))

# 2. polymer with the longest SMILES -- smiles_len is column 0
idx_longest = np.argmax(features[:, 0])
print(f"\nRow {idx_longest} has the longest SMILES ({features[idx_longest, 0]} chars)")
print("SMILES:", df['SMILES'].iloc[idx_longest])

# 3. sum of each feature for the first 100 polymers
print("\nSum of each feature over first 100 polymers:")
print(np.sum(features[:100], axis=0))

---
## 11.4 Broadcasting

Broadcasting is NumPy's mechanism for performing arithmetic on arrays of **different shapes**. The rule is simple:

> NumPy compares shapes element-wise, starting from the **trailing** dimensions. Two dimensions are compatible if they are equal, or one of them is 1.

A common use case: **standardizing** our feature matrix (subtracting the mean and dividing by the standard deviation). The feature matrix is `(7030, 6)` and the mean vector is `(6,)`. Can we subtract them directly?

In [ ]:
# Yes! Broadcasting stretches the (6,) vector to match (7030, 6)
features_standardized = (features - features.mean(axis=0)) / features.std(axis=0)

print("Original first row: ", features[0])
print("Standardized first row:", features_standardized[0])

In [ ]:
# Verify: the standardized features should have mean ~0 and std ~1
print("Means after standardization:", features_standardized.mean(axis=0).round(6))
print("Stds after standardization: ", features_standardized.std(axis=0).round(6))

### When broadcasting fails

If the shapes are not compatible, you get an error:

In [ ]:
# This will fail: shapes (3,) and (4,) are not compatible
try:
    np.array([1, 2, 3]) + np.array([10, 20, 30, 40])
except ValueError as e:
    print("Error:", e)

### Exercise 3

**Min-max normalization** scales each feature to the range [0, 1]:

$$X_{\text{norm}} = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$$

1. Compute `X_min` and `X_max` along `axis=0`.
2. Apply the formula using broadcasting.
3. Verify that the minimum of each column is 0 and the maximum is 1.

In [ ]:
# Your code here


In [ ]:
X_min = features.min(axis=0)
X_max = features.max(axis=0)

features_norm = (features - X_min) / (X_max - X_min)

print("Min per column (should be 0):", features_norm.min(axis=0))
print("Max per column (should be 1):", features_norm.max(axis=0))

---
## 11.5 Stacking and Concatenation

Often we compute a new feature and want to add it as a column to our feature matrix. NumPy provides several functions for combining arrays:

In [ ]:
# Suppose we compute a new feature: ratio of oxygen to carbon counts
o_to_c_ratio = features[:, 3] / np.maximum(features[:, 2], 1)  # avoid division by zero
print("New feature shape:", o_to_c_ratio.shape)  # (7030,)

In [ ]:
# Add it as a new column using np.column_stack
features_extended = np.column_stack([features, o_to_c_ratio])
print("Extended shape:", features_extended.shape)  # (7030, 7)

Other useful functions:
- `np.hstack` -- stack horizontally (same as `column_stack` for 2D)
- `np.vstack` -- stack vertically (add rows)
- `np.concatenate` -- general version, specify `axis`

In [ ]:
# Stack two arrays vertically (e.g. combine train and test data)
a = np.ones((3, 4))
b = np.zeros((2, 4))
c = np.vstack([a, b])
print(c)
print("Shape:", c.shape)  # (5, 4)

### `np.newaxis` -- adding a dimension

Sometimes you need to convert a 1D array `(n,)` to a column vector `(n, 1)` or row vector `(1, n)`. Use `np.newaxis`:

In [ ]:
v = np.array([1, 2, 3])
print("Original shape:", v.shape)           # (3,)
print("Column vector: ", v[:, np.newaxis].shape)  # (3, 1)
print("Row vector:    ", v[np.newaxis, :].shape)  # (1, 3)

---
## 11.6 Matrix Multiplication

Machine learning models compute predictions as:

$$\hat{y} = X \cdot w + b$$

where $X$ is the feature matrix, $w$ is a weight vector, and $b$ is a bias. The `@` operator (or `np.dot`) performs matrix multiplication.

In [ ]:
# Simple example: (2x3) @ (3x1) = (2x1)
A = np.array([[1, 2, 3],
              [4, 5, 6]])
w = np.array([[0.1],
              [0.2],
              [0.3]])

result = A @ w
print("A shape:", A.shape)
print("w shape:", w.shape)
print("Result: ", result.flatten())
print("Result shape:", result.shape)

Let's simulate a linear model on our polymer features:

In [ ]:
# Random weights (in a real model, these are learned from data)
np.random.seed(42)
w = np.random.randn(6, 1)  # 6 features -> 1 output
b = 0.35  # bias

# "Predict" FFV using our standardized features
predictions = features_standardized @ w + b

print("Predictions shape:", predictions.shape)
print("First 5 predictions:", predictions[:5].flatten())

In [ ]:
# Of course these are random -- but this is exactly what a LinearRegression does!
# In Notebook 13 we will let sklearn learn the optimal weights.
ffv = df["FFV"].values

plt.figure(figsize=(6, 4))
plt.scatter(ffv, predictions.flatten(), alpha=0.3, s=5)
plt.xlabel("Actual FFV")
plt.ylabel("Random Prediction")
plt.title("Random weights -- no real pattern (yet!)")
plt.tight_layout()
plt.show()

### Exercise 4

1. Compute the **correlation matrix** of the 6 features. The formula for the correlation of standardized data is:

$$\text{Corr} = \frac{1}{n} X_s^T \cdot X_s$$

   where $X_s$ is the standardized feature matrix.  
   Hint: use `features_standardized.T @ features_standardized / len(features_standardized)`

2. Display it using `plt.imshow(corr, cmap='coolwarm')` and add `plt.colorbar()`.
3. Which two features are most correlated?

In [ ]:
# Your code here


In [ ]:
n = len(features_standardized)
corr = features_standardized.T @ features_standardized / n

plt.figure(figsize=(6, 5))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='correlation')
plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha='right')
plt.yticks(range(len(feature_cols)), feature_cols)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()

# Find the most correlated feature pair (ignoring the diagonal)
corr_no_diag = corr.copy()
np.fill_diagonal(corr_no_diag, 0)
i, j = np.unravel_index(np.argmax(np.abs(corr_no_diag)), corr_no_diag.shape)
print(f"Most correlated pair: {feature_cols[i]} and {feature_cols[j]} "
      f"(corr = {corr[i, j]:.3f})")

---
## 11.7 Summary

| Concept | Key Functions |
|---------|---------------|
| Shape & reshape | `shape`, `ndim`, `reshape(-1, n)`, `flatten()`, `ravel()` |
| Aggregation | `np.mean`, `np.std`, `np.sum`, `np.min`, `np.max`, `np.argmax` + `axis` |
| Broadcasting | Arrays with compatible shapes are automatically "stretched" |
| Stacking | `np.column_stack`, `np.vstack`, `np.hstack`, `np.concatenate` |
| Matrix multiply | `@` operator, `np.dot` |